# 4.3 · Interpretabilidad y análisis de errores

**Tiempo estimado:** 15 min.

**Objetivos.**

1. **Recap SHAP** del notebook 3.2 sobre el modelo de árbol — qué features mandan en las crecidas.
2. **Análisis de errores por régimen** (estiaje vs crecida) y por mes.
3. Mensaje final: la **explicación del error** importa tanto como el error mismo.

## Mini-intro (5 min)

**SHAP en DL.** Funciona pero es **lento**: `shap.DeepExplainer` o `shap.GradientExplainer` para Keras. Para LSTMs largas, el coste es alto. Alternativas:

- **Integrated Gradients** (`tf-keras-vis`).
- **Permutation importance** (modelo-agnóstico, lento).
- **Attention rollouts** (si el modelo tiene atención).

En esta sesión usamos SHAP sobre LightGBM (rápido) — la interpretación es transferible al LSTM cualitativamente.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../sesion1'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import shap
from skforecast.recursive import ForecasterRecursive

import utils_datos as ud

plt.rcParams.update({'figure.figsize': (10, 3.4), 'axes.grid': True, 'grid.alpha': 0.3})

In [ ]:
caudal = ud.cargar_caudal_genil()
lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio='2010-01-01', fecha_fin='2020-12-31')
df = pd.DataFrame({'caudal': caudal, 'lluvia': lluvia}).loc['2011':'2020']
df = df.asfreq('D').interpolate('linear', limit=7).dropna()

def exog_diaria(df):
    out = pd.DataFrame(index=df.index)
    for w in (3, 7, 14, 30):
        out[f'p_acum{w}d'] = df['lluvia'].rolling(w).sum().shift(1)
    idx = out.index
    out['sin_an'] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    out['cos_an'] = np.cos(2 * np.pi * idx.dayofyear / 365.25)
    return out.asfreq('D')

X_exog = exog_diaria(df).dropna().asfreq('D')
y = df['caudal'].loc[X_exog.index].asfreq('D')
split = pd.Timestamp('2018-01-01')
y_tr, y_te = y.loc[:split].iloc[:-1].asfreq('D'), y.loc[split:].asfreq('D')
X_tr, X_te = X_exog.loc[y_tr.index].asfreq('D'), X_exog.loc[y_te.index].asfreq('D')

f = ForecasterRecursive(
    regressor=lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                 n_jobs=-1, verbose=-1, random_state=0),
    lags=[1, 2, 3, 7, 14, 30, 90, 365],
)
f.fit(y=y_tr, exog=X_tr)
pred = f.predict(steps=len(y_te), exog=X_te)
print(f'Modelo listo. RMSE test: {np.sqrt(((y_te-pred)**2).mean()):.3f}')

## 1 · SHAP en crecidas vs estiaje

Comparamos las contribuciones cuando el modelo predice un valor **alto** vs **bajo**.

In [ ]:
X_design, _ = f.create_train_X_y(
    y=pd.concat([y_tr.iloc[-365:], y_te]),
    exog=pd.concat([X_tr.iloc[-365:], X_te]),
)
X_design = X_design.loc[y_te.index]

explainer = shap.TreeExplainer(f.regressor)
sv = explainer(X_design)

# Top 10 días de mayor predicción y los de menor
top_alto = pred.nlargest(20).index
top_bajo = pred.nsmallest(20).index

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
shap.plots.bar(sv[X_design.index.isin(top_alto)], max_display=10, show=False, ax=axes[0])
axes[0].set_title('Crecidas (top 20 pred)')
shap.plots.bar(sv[X_design.index.isin(top_bajo)], max_display=10, show=False, ax=axes[1])
axes[1].set_title('Estiaje (bottom 20 pred)')
plt.tight_layout()

**Lectura típica:**

- En **crecidas** dominan `p_acum7d` y `p_acum14d` (lluvia acumulada reciente) — el modelo "ve" el evento por la lluvia.
- En **estiaje**, dominan `lag_1` y `cos_an` (estado actual + estación seca) — el modelo persiste.

## 2 · Errores por régimen

In [ ]:
df_err = pd.DataFrame({'obs': y_te.values, 'pred': pred.values}, index=y_te.index)
df_err['err'] = df_err['pred'] - df_err['obs']
df_err['mes'] = df_err.index.month
# Bins por cuartiles del observado (robusto a periodos sin crecida)
df_err['regimen'] = pd.qcut(df_err['obs'], q=4, labels=['estiaje', 'bajo', 'medio', 'alto'])

print('MAE y bias por régimen (cuartiles):')
tabla = df_err.groupby('regimen', observed=True).agg(
    n=('err', 'count'),
    obs_max=('obs', 'max'),
    bias=('err', 'mean'),
    mae=('err', lambda x: x.abs().mean()),
).round(3)
print(tabla)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
df_err.boxplot(column='err', by='regimen', ax=axes[0], showfliers=False)
axes[0].axhline(0, color='k', lw=0.5); axes[0].set_ylabel('error (pred - obs)')
df_err.boxplot(column='err', by='mes', ax=axes[1], showfliers=False)
axes[1].axhline(0, color='k', lw=0.5); axes[1].set_ylabel('error (pred - obs)')
for ax in axes: ax.set_title('')
plt.suptitle('')
plt.tight_layout()

**Diagnóstico esperado:**

- **Bias negativo en crecidas** = el modelo infraestima los picos. Firma de los modelos de árbol y LSTM con pocos datos.
- **Bias casi cero en medio y bajo** = el modelo persiste bien.
- **Por mes**: errores mayores en febrero-marzo (estación de avenidas) y agosto (estiaje extremo en cuenca regulada).

## 3 · Hidrograma del peor pico

In [ ]:
peor_pico = df_err.loc[df_err['regimen'] == 'alto', 'err'].abs().idxmax()
ventana = pd.date_range(peor_pico - pd.Timedelta('30D'), peor_pico + pd.Timedelta('15D'))

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(ventana, df_err.reindex(ventana)['obs'], color='black', lw=1.5, label='observado')
ax.plot(ventana, df_err.reindex(ventana)['pred'], color='#c2410c', lw=1.5, ls='--', label='predicho')
ax.axvline(peor_pico, color='red', lw=0.7, ls=':')
ax.set_ylabel('Q (m³/s)'); ax.legend()
ax.set_title(f'Hidrograma alrededor del peor error en cuartil alto: {peor_pico.date()}')
plt.tight_layout()

## 4 · Mensaje de cierre

Después de dos sesiones modelando, un buen análisis termina **siempre** con:

1. **¿Qué errores comete el modelo y por qué?** (SHAP + análisis por régimen).
2. **¿En qué situaciones NO se debe usar?** (típicamente: extrapolación a regímenes nuevos, eventos sin precedente).
3. **¿Qué mejorarías con más datos / más features?** (lluvia distribuida, niveles aguas arriba, deshielo).

Sin esto, lo que entregas es un *score*, no un modelo.

## 5 · Ejercicios

1. **SHAP sobre LSTM.** Usa `shap.GradientExplainer` con el modelo Keras del notebook 4.1. (Cuidado: lento. Subselecciona ~100 muestras.)
2. **Error vs lluvia.** Plotea el error del modelo (`err`) frente a `p_acum7d` del día. ¿Hay correlación visible?
3. **Quintil más alto.** Recalcula NSE/KGE sólo en el 20% más alto del observado. ¿Cuánto peor está el modelo en eventos extremos?
4. **Reto.** Implementa un análisis de errores **encadenado**: ¿los errores tienden a venir en rachas? Calcula la ACF del error y discute si hay autocorrelación residual.